## **Nugen Intelligence**
<img src="https://nugen.in/logo.png" alt="Nugen Logo" width="200"/>

Domain-aligned foundational models at industry leading speeds and zero-data retention! To learn more, visit [Nugen](https://docs.nugen.in/introduction)

**Step 1**

Install the required Python packages

In [ ]:
!pip install --quiet requests pandas python-dotenv

Import Required Libraries

In [ ]:
import os 
import requests
import pandas as pd
import json
import time
from dotenv import load_dotenv
load_dotenv()

Set up the Nugen API Client

To read more about Nugen API and access free API keys, you can visit Nugen Dashboard

Configure the API Client

Set your Nugen API key.

In [ ]:
api_key = os.getenv("NUGEN_API_KEY")

In [ ]:
headers = {"Authorization": f"Bearer {api_key}"}

Here, we define the API base URL and your API key. Replace <--nugen api key--> with your actual key to authenticate your requests to the Nugen API. The MODEL variable specifies the model we will use for generating the routines.

In [ ]:
BASE_URL = "https://api.nugen.in"

**Step 2**

### **Upload Dataset**

Upload your dataset

In [ ]:
with open("your_dataset.jsonl", "rb") as f:
    files = {
        "files": ("your_dataset.jsonl", f, "application/json")
    }

    data = {
        "categories": "text/json"
    }

    document_response = requests.post(
        url,
        headers=headers,
        data=data,
        files=files
    )
print(document_response.text)

### **Get status of Document uploaded**

Once the upload is complete, retrieve the document ID

This endpoint returns metadata such as the document ID and processing status.

In [ ]:
response_data = json.loads(document_response.text) 

id = response_data["document_ids"][0]

In [ ]:
document_status_url = f"{BASE_URL}/api/v3/documents/{id}"

In [ ]:
document_status_response = requests.get(document_status_url, headers=headers)

print(document_status_response.text)

### **Generate Benchmark**

Note: Benchmark questions are generated using uploaded dataset

In [ ]:
response_data = json.loads(document_status_response.text) 

document_id = response_data["document_id"]

In [ ]:
generate_benchmark_url = f"{BASE_URL}/api/v3/benchmark/create"

In [ ]:
payload = {
    "documents": [document_id],
    "num_questions": 20
}

benchmark_response = requests.post(generate_benchmark_url, json=payload, headers=headers)

print(benchmark_response.text)

### **Get status Benchmark Generation**

Check whether benchmark generation has completed.

In [ ]:
response_data = json.loads(benchmark_response.text)

benchmark_id = response_data["benchmark_id"]

In [ ]:
benchmark_status_url = f"{BASE_URL}/api/v3/benchmark/status/{benchmark_id}"

In [ ]:
benckmark_status_response = requests.get(benchmark_status_url, headers=headers)

print(benckmark_status_response.text)

### **Get Benchmark Data**

In [ ]:
response_data = json.loads(benckmark_status_response.text)

benchmark_id = response_data["benchmark_id"]

In [ ]:
benchmark_data_url = f"{BASE_URL}/api/v3/benchmark/status/{benchmark_id}/data"

In [ ]:
response = requests.get(benchmark_data_url, headers=headers)

print(response.text)

Retrieve complete benchmark data with all questions and answers.

### **Create Alignment Project**

In [ ]:
alignment_url = f"{BASE_URL}/api/v3/alignment-project/create"

In [ ]:
payload = {
    "name": "My Alignment Project ",
    "base_model": "llama-v3p2-3b-reasoning",
    "document_ids": [document_id],
    "workflow_id": "workflow-abc123",
    "benchmark_id": benchmark_id,
    "description": "This project aims to align the model for better vision alignment."
}

print(payload)
alignment_response = requests.post(alignment_url, json=payload, headers=headers)

print(alignment_response.text)

### **Check Alignment Status**

Track the alignment status.

In [ ]:
response_data = json.loads(alignment_response.text)

alignment_id = response_data["alignment_id"]

In [ ]:
alignment_status_url = f"{BASE_URL}/api/v3/alignment-project/status/{alignment_id}"

In [ ]:
while True:
    alignment_response = requests.get(alignment_status_url, headers=headers)
    alignment_response.raise_for_status()
    print(alignment_response.text)
    data = alignment_response.json()
    status = data["status"]

    print(f"Current status of Alignment: {status}")

    if status == "READY":
        print("Alignment completed.")
        break

    if status == "FAILED":
        raise Exception("Alignment failed.")
    time.sleep(10)

### **Deploy Aligned Model**

Once alignment completes, you'll be able to deploy the model.

In [ ]:
response_data = json.loads(alignment_response.text)

model_id = response_data["data"]["model_id"]

In [ ]:
deploy_url = f"{BASE_URL}/api/v3/models/deploy-model/{model_id}"

In [ ]:
deploy_response = requests.post(deploy_url, headers=headers)

print(deploy_response.text)

### **Deploy Status**

Check the deployment status of an aligned model.

In [ ]:
response_data = json.loads(deploy_response.text)

model_id = response_data["model_id"]

In [ ]:
deploy_status_url = f"https://api.nugen.in/api/v3/models/deploy-model/{model_id}/status"

In [ ]:
response = requests.get(deploy_status_url, headers=headers)

print(response.text)

### **Run Inference**

Once alignment completes, you'll receive an Aligned Model ID. Use this model for inference.

In [ ]:
payload = {
    "model": model_id,
    "messages": [
        {
            "role": "system",
            "content": "Tell me about India"
        }
    ],
    "max_tokens": 100,
    "prompt_truncate_len": 123,
    "temperature": 1,
    "stream": False,
}

In [ ]:
response = requests.post(url, headers=headers, json=payload)

print(response.text)